# Homework Assignment: Advanced LLM Acceleration: Speculative Decoding & Quantization

**Target hardware:** 1x NVIDIA H100 80GB

**Base model:** `Qwen/Qwen3-8B`

## Objective

Modern LLM deployment is often limited by memory bandwidth, verifier cost, and decoding overhead. In this lab, you will build and evaluate a multi-stage acceleration pipeline for `Qwen/Qwen3-8B`:

- train an EAGLE-3 speculative decoding draft head;
- quantize the verifier model with FP8 dynamic quantization;
- benchmark baseline, speculative decoding, quantization, and the combined setup;
- explain which optimization should be applied first and why.

The main question for the assignment:

**Which should be done first for this workflow: speculative decoding training or quantization?**

Your answer must be supported by the training setup, benchmark results, and a short technical explanation.

---

## References

- Speculators library: <https://github.com/vllm-project/speculators>
- Offline EAGLE-3 training tutorial: <https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train_eagle3_offline>
- FP8 dynamic quantization reference: <https://github.com/vllm-project/llm-compressor/blob/main/examples/quantization_w8a8_fp8/README.md>

---

## Required Library Versions

Use separate environments. The speculators training workflow and vLLM serving workflow have dependency conflicts, so a single environment is not expected to work cleanly.

| Component | Required version / constraint | Purpose | venv |
| --- | --- | --- | --- |
| `speculators` | Git tag `v0.5.0`, editable install | Data preparation, hidden-state generation, EAGLE-3 training | `speculators_venv` |
| `vllm` | `v0.20.0` | Serving and benchmark runtime | `vllm_venv` |
| `fastapi` | `<0.137` | Compatibility with the selected vLLM version | `vllm_venv` |
| `llmcompressor` | `v0.12.0` | FP8 dynamic quantization | `comp_venv` |
| Python for vLLM / quantization | Python `3.12` recommended | Reproducible environment setup | --- |
| Model | `Qwen/Qwen3-8B` | Verifier model | --- |
| Dataset | ShareGPT-style conversations | Training data source | --- |

Expected local environments:

- `speculators_venv`
- `vllm_venv`
- `comp_venv`

Do not submit the virtual environments.

Note: To install `speculators`, clone the repository and install it in editable mode in `speculators_venv`.

---

## Task 1: Environment & Data Orchestration

Set up the training and serving environments, then prepare the ShareGPT data for offline EAGLE-3 training.

Required work:

- create isolated environments for Speculators, vLLM, and quantization;
- install the required versions listed above;
- prepare ShareGPT-style data for `Qwen/Qwen3-8B`;
- generate hidden states for offline EAGLE-3 training.

Background:

Offline EAGLE-3 training means the verifier model hidden states are precomputed before training the draft head. This lets the workflow run sequentially on one GPU, but it uses much more disk space.

Online EAGLE-3 training computes hidden states during training. It can be faster end to end, but it normally requires at least two GPUs: one for inference/data generation and one for training.

Use the Speculators offline EAGLE-3 tutorial as the main workflow reference:

<https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train_eagle3_offline>

You need to implement an offline EAGLE-3 training pipeline.

Hints:

- Watch local disk usage. A few thousand samples can already require around 140GB after hidden states are generated.
- A reasonable starting point is `max-samples=3000` and sequence length `2048`.
- If you need better draft-head quality, increasing the number of samples is more useful than changing many settings at once.
- Use the `scripts/launch_vllm.py` script to run the vLLM server.

Question: Why do hidden states require much more disk space than the original text dataset?

Troubleshooting hints:

- If hidden-state generation reports missing temporary files, clear stale temporary hidden-state files (`/tmp/hidden_states/*`) and rerun generation.
- If hidden-state sequence lengths do not match tokenized sequence lengths, verify the vLLM version first.
- If disk usage grows too quickly, reduce the number of samples before changing other parameters.

---

### Task 1 — Answer

**Run result:** offline hidden-state generation completed for `max-samples=3000` at sequence length `2048`. The generated hidden states occupy **~108.3 GB** on disk, and `--validate-outputs` confirmed that hidden-state sequence lengths match the tokenized sequence lengths (correct vLLM version).

**Q: Why do hidden states require much more disk space than the original text dataset?**

Because text and hidden states store *fundamentally different objects per token*:

- **Text / token IDs are compact and discrete.** A tokenized sequence stores one integer ID per token (≈2–4 bytes). For 3000 samples × 2048 tokens ≈ 6.1M tokens, that is only tens of MB. The raw ShareGPT text is likewise a few hundred MB at most.
- **Hidden states are dense, high-dimensional, continuous tensors.** For *every* token we save a hidden vector of size `hidden_size = 4096` in a multi-byte float format (bf16 = 2 bytes). EAGLE-3 additionally needs hidden states from **several layers** of the verifier (the low/mid/high auxiliary hidden states that form the draft-head input, plus the final-layer target hidden state). So each token costs on the order of thousands of floats instead of one integer.

Concretely: 108.3 GB ÷ 6.1M tokens ≈ **17.6 KB per token** ≈ ~8.8k bf16 values per token ≈ roughly two 4096-dim hidden vectors per token — versus ~4 bytes for the corresponding token ID. That is a **~1000× per-token blow-up**.

In short: disk usage scales as `num_tokens × hidden_dim × num_stored_layers × bytes_per_value`, whereas text scales as `num_tokens × bytes_per_token_id`. The `hidden_dim × num_stored_layers × bytes_per_value` factor is what makes hidden states enormous, and it is also why the Task 1 hint warns that a few thousand samples can already require ~140 GB.

---

### Task 2 — Answer

**Training setup and data-size ablation.** We trained the EAGLE-3 head twice. The first run used `max-samples=3000` (as suggested in Task 1); its draft head reached only ~1209 tok/s in serving (below the 1250 threshold, see Task 4). Following the Task 1 hint that *"increasing the number of samples is more useful than changing many settings at once"*, we regenerated hidden states at `max-samples=5000` (same seq-length 2048) and retrained. This is the checkpoint used for all serving/benchmarks.

**Final validation metrics (retrained on 5000 samples, epoch index 6 = 7th epoch):**

| Metric | 3000 samples | **5000 samples (final)** | Reference |
| --- | ---: | ---: | ---: |
| `val/loss_0_epoch` | `2.988` | **`2.437`** | `2.509` |
| `val/full_acc_0_epoch` | `0.436` | **`0.500`** | `0.463` |
| `val/cond_acc_0_epoch` | `0.436` | **`0.500`** | `0.463` |
| `val/loss_1_epoch` | `4.321` | **`3.785`** | `3.778` |
| `val/full_acc_1_epoch` | `0.154` | **`0.203`** | `0.181` |
| `val/cond_acc_1_epoch` | `0.342` | **`0.396`** | `0.364` |
| `val/loss_2_epoch` | `5.103` | **`4.633`** | `4.550` |
| `val/full_acc_2_epoch` | `0.050` | **`0.076`** | `0.069` |
| `val/cond_acc_2_epoch` | `0.298` | **`0.349`** | `0.320` |
| `val/loss_epoch` | `12.412` | **`10.856`** | `10.837` |

More data moved every metric in the right direction and pushed `full_acc_0` from `0.436` to **`0.500`**, now *above* the reference `0.463`. This directly validates the assignment's guidance: for a weak draft head, adding samples is the highest-leverage change. Checkpoints `0..6` plus **`checkpoint_best`** (lowest validation loss) were saved under `output/checkpoints/`; `checkpoint_best` is used for serving/benchmarking.

**Q1: What do `full_acc` and `cond_acc` measure in speculative decoding training?**

They are per-draft-position acceptance-accuracy metrics for the EAGLE-3 head:

- **`full_acc_k` (full / unconditional accuracy at position `k`)** — the probability that the *whole* draft chain up to and including position `k` is correct, i.e. that positions `0..k` would all be accepted by the verifier in a free-running rollout. This is the quantity that directly governs **acceptance length**.
- **`cond_acc_k` (conditional accuracy at position `k`)** — the per-step probability that position `k` is correct *given that positions `0..k-1` were already correct*. It isolates the intrinsic difficulty of predicting one more step, independent of upstream errors.

At position 0 there is nothing to condition on, so `full_acc_0 = cond_acc_0 = 0.500`. The two are linked multiplicatively, and our final numbers confirm it almost exactly:

- `full_acc_1 ≈ cond_acc_0 × cond_acc_1 = 0.500 × 0.396 = 0.198` (measured `0.203`)
- `full_acc_2 ≈ 0.500 × 0.396 × 0.349 = 0.069` (measured `0.076`)

So `full_acc_k ≈ Π cond_acc_j`, which is exactly why full accuracy collapses much faster than conditional accuracy.

**Q2: Why does accuracy usually decrease for later speculative positions?**

Two compounding reasons:

1. **Error propagation (the dominant effect on `full_acc`).** Acceptance of position `k` requires *every* earlier position to also be correct. Since each `cond_acc_j < 1`, the joint probability is a shrinking product — `full_acc` decays roughly geometrically (0.500 → 0.203 → 0.076).
2. **Growing intrinsic difficulty (visible even in `cond_acc`).** `cond_acc` also declines (0.500 → 0.396 → 0.349) because the draft head must predict tokens further into the future from a single set of verifier hidden states, without receiving fresh verifier context per step. The token distribution further ahead is more uncertain and multi-modal, so each additional step is genuinely harder.

**Q3: What would you change if the first-position accuracy is very low?**

Fix **data generation (Task 1) first, not the training recipe.** A low `full_acc_0` almost always means the draft-head inputs are misaligned with the target tokens, e.g.:

- wrong/mismatched **vLLM version** so hidden-state seq-len doesn't match token count (off-by-one / misalignment between aux hidden states and target tokens);
- **stale `/tmp/hidden_states/*`** files mixed into the run;
- wrong verifier layers captured, or a **chat template / tokenizer** that doesn't match `Qwen/Qwen3-8B`.

Only after confirming the data is aligned should you touch training. Here `full_acc_0` was already healthy (0.436), so the data was fine — but it was still *sub-threshold* for serving, and the right lever was **more training samples** (3000 → 5000), which raised `full_acc_0` to 0.500 and cleared the serving throughput bar (Task 4). More epochs / learning-rate tuning come after that.

---

### Task 3 — Answer

**Run result:** `Qwen3-8B-FP8-Dynamic` was produced with `llmcompressor`; the base `Qwen/Qwen3-8B` was left untouched. The saved `config.json` contains a valid quantization section that matches every expected property:

| Property | Expected | Config value |
| --- | --- | --- |
| Quantization method | compressed tensors | `quant_method: "compressed-tensors"`, `format: "float-quantized"` |
| Weight format | FP8 | `weights: {num_bits: 8, type: "float", strategy: "channel", dynamic: false}` |
| Activation format | dynamic FP8 | `input_activations: {num_bits: 8, type: "float", strategy: "token", dynamic: true}` |
| Target modules | linear layers | `targets: ["Linear"]` |
| Ignored module | `lm_head` | `ignore: ["lm_head"]` |

So this is **W8A8-FP8**: weights are static per-channel FP8, activations are dynamic per-token FP8 (scales computed at runtime, no activation calibration set needed), `lm_head` kept in BF16.

**Q1: Why is FP8 dynamic quantization useful for serving on H100?**

- **Native hardware support.** Hopper (H100) has FP8 (E4M3) tensor cores that run GEMMs at roughly **2× the throughput of BF16**. FP8 W8A8 actually executes the matmuls in FP8, so the speedup is real compute, not just storage.
- **Memory bandwidth is the decode bottleneck.** Autoregressive decoding is memory-bandwidth bound: each token requires streaming all weights from HBM. FP8 **halves the weight bytes** moved per token vs BF16, directly lowering TPOT and raising output token throughput. It also frees HBM for a larger KV cache / more concurrency.
- **Dynamic activation quant preserves accuracy cheaply.** Per-token dynamic scales adapt to each activation's range at runtime, so no calibration dataset is required for activations and outliers are handled per-token — you get FP8 speed with minimal accuracy loss.

**Q2: Why might `lm_head` be excluded from quantization?**

- **It directly shapes the output distribution.** `lm_head` maps the final hidden state to logits over the full vocabulary (~151k for Qwen3). Any quantization error here perturbs the logits/softmax that decide the actual token probabilities, so it disproportionately hurts perplexity/quality — and, for speculative decoding, it perturbs the very distribution the draft is verified against (see Q3).
- **Little to gain, much to lose.** It is a single projection run once per token, negligible compared to the 36 transformer blocks, so quantizing it yields almost no speedup while risking output quality. Keeping it in BF16 is the standard, favorable trade-off.

**Q3: How can quantization affect speculative decoding acceptance rate?**

Acceptance depends on the **verifier's output distribution**, and quantizing the verifier shifts that distribution:

- The EAGLE-3 draft head was trained against hidden states / logits from the **BF16** verifier. Serving with an FP8 verifier slightly changes its logits, so a draft token that the BF16 verifier would have accepted may now be rejected (or vice-versa). In general this can **lower** per-token acceptance, and it is one reason the draft-token count should be **re-tuned** after quantization rather than reused.
- **Caution comparing the headline numbers:** in the reference the FP8+spec run shows a *higher* acceptance rate (36.50%) than BF16+spec (22.48%), but those used **different draft-token counts (1 vs 2)**. Acceptance rate = accepted / drafted; with only 1 draft token every proposal is an easy position-0 prediction (highest `cond_acc`), so the average is naturally higher, whereas with 2 draft tokens half the proposals are the harder position-1 prediction, dragging the mean down. So that difference is mostly an artifact of draft-token count, not quantization improving draft quality — which is exactly why Task 4 tunes the draft-token count separately per configuration.

---

# Task 2: Train an EAGLE-3 Draft Head

Train an EAGLE-3 speculative decoding draft head using the precomputed hidden states.

Required work:

- Use `Qwen/Qwen3-8B` as the verifier model for training.
- Save checkpoints under `output/checkpoints/`.
- Use the best checkpoint for serving and benchmarking.
- Track validation loss and acceptance-related accuracy metrics across draft positions.

Hints:

- If first-position accuracy is very low, inspect data generation before changing the training recipe.
- Later speculative positions are harder, so compare position-wise metrics instead of relying only on total loss.

Reference training result from one run:

| Metric | Value |
| --- | ---: |
| `val/loss_0_epoch` | `2.509` |
| `val/full_acc_0_epoch` | `0.463` |
| `val/cond_acc_0_epoch` | `0.463` |
| `val/loss_1_epoch` | `3.778` |
| `val/full_acc_1_epoch` | `0.181` |
| `val/cond_acc_1_epoch` | `0.364` |
| `val/loss_2_epoch` | `4.550` |
| `val/full_acc_2_epoch` | `0.069` |
| `val/cond_acc_2_epoch` | `0.320` |
| `val/loss_epoch` | `10.837` |
| Epoch | `4` |

Note: Epoch=4 means this is the 5th epoch, the index starts with 0.

Questions to answer:

1. What do `full_acc` and `cond_acc` measure in speculative decoding training?
2. Why does accuracy usually decrease for later speculative positions?
3. What would you change if the first-position accuracy is very low?

---

## Task 3: Quantize the Verifier Model

Quantize `Qwen/Qwen3-8B` using FP8 dynamic quantization.

Required work:

- Use `llmcompressor`.
- Apply FP8 dynamic quantization to linear layers.
- Keep `lm_head` unquantized.
- Save the quantized model as a separate local model directory, for example `Qwen3-8B-FP8-Dynamic`.
- Do not overwrite the original model checkpoint.

Reference material:

<https://github.com/vllm-project/llm-compressor/blob/main/examples/quantization_w8a8_fp8/README.md>

Hints:

- Verify the saved model config contains a quantization section before benchmarking.
- Keep the original BF16 model available so you can compare baseline and quantized serving behavior.

Expected quantization properties:

| Property | Expected value |
| --- | --- |
| Quantization method | compressed tensors |
| Weight format | FP8 |
| Activation format | dynamic FP8 |
| Target modules | linear layers |
| Ignored module | `lm_head` |

Questions to answer:

1. Why is FP8 dynamic quantization useful for serving on H100?
2. Why might `lm_head` be excluded from quantization?
3. How can quantization affect speculative decoding acceptance rate?

---

## Task 4: Serve and Benchmark

Benchmark four configurations:

1. Baseline `Qwen/Qwen3-8B`
2. `Qwen/Qwen3-8B` with the trained EAGLE-3 draft head
3. FP8 dynamically quantized `Qwen3-8B`
4. FP8 dynamically quantized `Qwen3-8B` with the trained EAGLE-3 draft head

Required work:

- Use the same benchmark dataset and benchmark settings for all four configurations.
- Keep concurrency fixed across experiments.
- Disable prefix caching unless you intentionally study its effect.
- Use a fixed seed where possible.
- Tune the number of speculative draft tokens separately for speculative decoding and FP8 + speculative decoding.

Important:

The reference results used different numbers of speculative draft tokens for the unquantized speculative-decoding run and the FP8 + speculative-decoding run. Do not assume one value is optimal for both. Tune it yourself and justify the final choice using throughput, acceptance rate, acceptance length, and TPOT.

Hints:

- Start with a small number of speculative tokens, then increase only if the acceptance behavior supports it.
- Compare output token throughput first, then use TTFT, TPOT, and acceptance metrics to explain the result.
- If a setting produces more draft work but little accepted work, it is probably not the best setting.

Script for benchmarking:

```bash
vllm bench serve \
    --model Qwen/Qwen3-8B \
    --dataset-name hf \
    --max-concurrency 8 \
    --dataset-path philschmid/mt-bench \
    --num-prompts 80
```


Reference benchmark results:

| Configuration | Duration, s | Requests/s | Output tok/s | Total tok/s | Mean TTFT, ms | Mean TPOT, ms | Acceptance rate |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Baseline | `24.35` | `3.29` | `841.22` | `1090.87` | `576.17` | `7.28` | N/A |
| Speculative decoding | `16.27` | `4.92` | `1258.65` | `1632.19` | `78.17` | `5.76` | `22.48%` |
| FP8 quantization | `13.06` | `6.12` | `1566.56` | `2031.82` | `51.18` | `4.90` | N/A |
| FP8 + speculative decoding | `11.59` | `6.90` | `1766.55` | `2290.82` | `30.24` | `4.28` | `36.50%` |

Reference speculative decoding details:

| Configuration | Reference draft tokens | Acceptance length | Drafts | Draft tokens | Accepted tokens |
| --- | ---: | ---: | ---: | ---: | ---: |
| Speculative decoding | `2` | `1.45` | `14088` | `28176` | `6334` |
| FP8 + speculative decoding | `1` | `1.36` | `14954` | `14954` | `5458` |

Your exact numbers may differ, but the relative comparison should be explainable.

Questions to answer:

1. Why can speculative decoding improve throughput even when acceptance rate is not close to 100%?
2. How many speculative tokens are optimal for this setup? Explain using acceptance rate, acceptance length, and TPOT.

---

## Final Report Requirements

Add serving benchmark results for three configurations to your final submission Jupyter notebook as text:

- speculative decoding;
- FP8 quantization;
- FP8 + speculative decoding.

---

## Scoring Rubric

Scores are based on `Output token throughput (tok/s)` from `vllm bench serve`.
Each row is pass/fail: if the threshold is not reached, that row receives 0 points.

| Requirement | Passing threshold | Points |
| --- | ---: | ---: |
| Speculative decoding result with trained EAGLE-3 draft head | `> 1250 tok/s` | 25 |
| FP8 dynamic quantization result | `> 1550 tok/s` | 10 |
| Best combined FP8 + speculative decoding result, with draft-token tuning and explanation | `> 1750 tok/s` | 15 |
| **Total** |  | **50** |




### Task 4 — Answer

**Summary of all four configurations (all measured warm, same dataset / concurrency 8 / seed 0 / prefix caching off):**

| Configuration | Draft tokens | Output tok/s | TPOT, ms | Mean TTFT, ms | Acceptance rate | Acceptance length | Rubric |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | :---: |
| Baseline (BF16) | — | 1114.85 | 7.00 | 51.32 | — | — | — |
| Speculative decoding (BF16 + EAGLE-3) | **1** | **1266.09** | 5.90 | 59.17 | 40.93% | 1.41 | ✅ > 1250 |
| FP8 quantization | — | **1595.76** | 4.88 | 37.60 | — | — | ✅ > 1550 |
| FP8 + speculative decoding | **1** | **1650.67** | 4.41 | 64.70 | 39.84% | 1.40 | ⚠ < 1750 |

(For reference, the notebook's "out-of-the-box" baseline example was measured cold at 841 tok/s with a 576 ms mean TTFT; our warm baseline is 1114 tok/s. All four rows above are warm so the comparison between them is apples-to-apples.)

**Q1: Why can speculative decoding improve throughput even when acceptance rate is not close to 100%?**

Because the verifier validates *all* proposed draft tokens in a **single parallel forward pass**, and decode is memory-bandwidth bound — one forward pass streams the model weights from HBM exactly once regardless of how many tokens it checks. So the cost is ~one verifier step but the output can be several tokens. What matters is not the acceptance *rate* but the **acceptance length** (mean accepted tokens per verifier step): as long as it is `> 1` and drafting is cheap relative to the verifier, you emit more tokens per unit of verifier bandwidth, so TPOT drops and throughput rises. The verifier's own next token is always correct, so acceptance length is `≥ 1` even if every draft is rejected — speculative decoding never *reduces* the tokens produced per verify step, it only risks spending extra draft compute. In our BF16 run the acceptance rate is only 40.93%, yet acceptance length 1.41 still cuts TPOT from 7.00 to 5.90 ms and lifts throughput from 1114 to 1266 tok/s.

**Q2: How many speculative tokens are optimal for this setup? Explain using acceptance rate, acceptance length, and TPOT.**

**One draft token (`num_speculative_tokens = 1`) is optimal for both the BF16 and the FP8 verifier**, tuned separately (sweeps in the two result cells above). Output throughput decreases monotonically as the draft-token count grows — BF16: 1266 → 1228 → 1157 → 1082; FP8: 1650 → 1554 → 1439. The reason is the shape of per-position acceptance: it collapses right after position 0 (pos0 ≈ 40% → pos1 ≈ 11% → pos2 ≈ 1%). Going from k=1 to k=2 raises acceptance *length* only marginally (1.41 → 1.52) while **doubling** the draft-head forward passes and the verification width (draft tokens 14.5k → 27k), so the added compute is not repaid — TPOT stays flat (BF16: 5.90 vs 5.89 ms) and wall-clock throughput falls. On the **FP8** verifier the effect is sharper: a decode step is already so cheap (TPOT 4.88 ms) that the draft overhead at k=2 makes it (1554 tok/s) *slower than FP8 with no draft head at all* (1595 tok/s); only k=1 remains a net win. This is exactly why the reference tuned BF16→2 but FP8→1: the optimum depends on how expensive the verifier step is, so it must be re-tuned per precision.

---

This is an example of the output we expect to be submitted. This is the result we get from the baseline model out of the box:
```
============ Serving Benchmark Result ============
Successful requests:                     80        
Failed requests:                         0         
Maximum request concurrency:             8         
Benchmark duration (s):                  24.35     
Total input tokens:                      6078      
Total generated tokens:                  20480     
Request throughput (req/s):              3.29      
Output token throughput (tok/s):         841.22    
Peak output token throughput (tok/s):    1144.00   
Peak concurrent requests:                16.00     
Total token throughput (tok/s):          1090.87   
---------------Time to First Token----------------
Mean TTFT (ms):                          576.17    
Median TTFT (ms):                        46.05     
P99 TTFT (ms):                           5677.04   
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          7.28      
Median TPOT (ms):                        7.01      
P99 TPOT (ms):                           11.66     
---------------Inter-token Latency----------------
Mean ITL (ms):                           7.28      
Median ITL (ms):                         7.00      
P99 ITL (ms):                            7.68      
==================================================

```

Speculative decoding benchmark results:

Configuration: BF16 `Qwen/Qwen3-8B` + trained EAGLE-3 draft head (5000-sample checkpoint), `num_speculative_tokens = 1` (tuned — see table below), warm run, prefix caching disabled, `max-concurrency 8`, `num-prompts 80`, `philschmid/mt-bench`.

```
============ Serving Benchmark Result ============
Successful requests:                     80        
Failed requests:                         0         
Maximum request concurrency:             8         
Benchmark duration (s):                  16.18     
Total input tokens:                      6078      
Total generated tokens:                  20480     
Request throughput (req/s):              4.95      
Output token throughput (tok/s):         1266.09   
Peak output token throughput (tok/s):    976.00    
Peak concurrent requests:                16.00     
Total token throughput (tok/s):          1641.84   
---------------Time to First Token----------------
Mean TTFT (ms):                          59.17     
Median TTFT (ms):                        25.60     
P99 TTFT (ms):                           361.27    
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          5.90      
Median TPOT (ms):                        5.86      
P99 TPOT (ms):                           6.63      
---------------Inter-token Latency----------------
Mean ITL (ms):                           8.30      
Median ITL (ms):                         8.24      
P99 ITL (ms):                            9.18      
---------------Speculative Decoding---------------
Acceptance rate (%):                     40.93     
Acceptance length:                       1.41      
Drafts:                                  14494     
Draft tokens:                            14494     
Accepted tokens:                         5932      
Per-position acceptance (%):
  Position 0:                            40.93     
==================================================
```

**Result: 1266.09 tok/s > 1250 threshold ✅** (vs. warm baseline ~1114 tok/s and reference 1258.65). TPOT drops from the baseline 7.00 ms to 5.90 ms (−16%).

**Draft-token tuning (BF16 + EAGLE-3), warm sweep:**

| `num_spec_tokens` | Output tok/s | TPOT, ms | Acceptance rate | Acceptance length | Per-position acceptance |
| ---: | ---: | ---: | ---: | ---: | --- |
| **1** | **1266.09** | 5.90 | 40.93% | 1.41 | pos0 40.93% |
| 2 | 1228.49 | 5.89 | 25.87% | 1.52 | pos0 40.35% / pos1 11.39% |
| 3 | 1156.76 | 6.34 | — | — | — |
| 4 | 1082.15 | 6.77 | — | — | — |

**Chosen: `num_speculative_tokens = 1`.** Output throughput is monotonically decreasing in the draft-token count (1266 → 1228 → 1157 → 1082). Going from k=1 to k=2 raises acceptance *length* slightly (1.41 → 1.52), but per-position acceptance collapses after position 0 (pos0 ≈ 41% → pos1 ≈ 11%), so the extra draft token doubles the draft-head forward passes and verification width (14.5k → 26.9k draft tokens) while adding little accepted work — classic "more draft work, little accepted work". TPOT is essentially flat between k=1 and k=2 (5.90 vs 5.89 ms), so the extra draft compute is not repaid and wall-clock throughput drops. Hence k=1 is optimal for the unquantized verifier.

FP8 quantization benchmark results:

Configuration: `Qwen3-8B-FP8-Dynamic` (W8A8 FP8 dynamic, `lm_head` in BF16), no speculative decoding, warm run, prefix caching disabled, `max-concurrency 8`, `num-prompts 80`, `philschmid/mt-bench`.

```
============ Serving Benchmark Result ============
Successful requests:                     80        
Failed requests:                         0         
Maximum request concurrency:             8         
Benchmark duration (s):                  12.83     
Total input tokens:                      6078      
Total generated tokens:                  20480     
Request throughput (req/s):              6.23      
Output token throughput (tok/s):         1595.76   
Peak output token throughput (tok/s):    1648.00   
Peak concurrent requests:                16.00     
Total token throughput (tok/s):          2069.35   
---------------Time to First Token----------------
Mean TTFT (ms):                          37.60     
Median TTFT (ms):                        25.25     
P99 TTFT (ms):                           136.89    
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          4.88      
Median TPOT (ms):                        4.88      
P99 TPOT (ms):                           4.95      
---------------Inter-token Latency----------------
Mean ITL (ms):                           4.88      
Median ITL (ms):                         4.87      
P99 ITL (ms):                            5.33      
==================================================
```

**Result: 1595.76 tok/s > 1550 threshold ✅** (reference 1566.56). FP8 alone beats the tuned BF16 speculative-decoding run (1266 tok/s) with **no draft head at all**. The gain is pure per-token acceleration: TPOT falls to **4.88 ms** (baseline 7.00 ms, −30%), because FP8 halves the weight bytes streamed from HBM per token and uses the H100's FP8 tensor cores. Note TPOT here is rock-steady (mean 4.88, P99 4.95) since every step does the same work — unlike speculative decoding, whose per-step cost varies with accept/reject.

## Final Conclusion — Which should be done first: speculative decoding training or quantization?

**Answer: do FP8 quantization first, then train / tune speculative decoding on top of the quantized verifier.**

The benchmark results and the training setup support this on three grounds:

**1. Cost vs. reward — quantization is cheap and delivers the bigger single-config win.**
FP8 dynamic quantization is a one-shot post-training step (minutes, no training data, no hidden-state generation, no GPU-hours) and it took throughput from a warm baseline of 1114 to **1595 tok/s (TPOT 7.00 → 4.88 ms)**. Speculative decoding, by contrast, required ~108–170 GB of hidden states, hours of offline generation, EAGLE-3 training, *and* a retrain (3000 → 5000 samples) just to clear its threshold — for a smaller BF16 gain (1114 → **1266 tok/s**). FP8 alone (1595) beats BF16 + speculative decoding (1266). If you can only do one, quantization wins on effort-adjusted throughput by a wide margin.

**2. Dependency direction — the draft head depends on the verifier, not the other way around.**
An EAGLE-3 draft head is trained against the verifier's hidden states and output distribution. If you train it on the BF16 model and *then* quantize, the verifier's distribution shifts and the acceptance behavior can change, so you may have to re-validate or retrain. Quantizing first fixes the verifier in its final production form, so the draft head is trained and tuned against exactly the model it will run with. There is no analogous dependency in the other direction — quantization does not care whether a draft head exists.

**3. Tuning depends on the final verifier precision.**
Our sweeps show the optimal `num_speculative_tokens` and even *whether speculative decoding is worth it* depend on how cheap the verifier step is. On the fast FP8 verifier the decode step is already ~4.9 ms, so speculative decoding adds only a little (1595 → **1650 tok/s**, +3.4%) and k=2 is actually *slower than FP8 with no draft head at all*. You can only make these tuning decisions correctly once the verifier is in its final (quantized) state — another reason to quantize first and treat speculative decoding as the last, verifier-specific optimization.

**Practical recommendation:** apply FP8 quantization first (largest, cheapest, dependency-free win and it sets the final verifier), then generate hidden states from the **quantized** verifier and train + tune the EAGLE-3 draft head against it, and finally re-tune the draft-token count for the FP8 verifier (here: `num_speculative_tokens = 1`).

The FP8 + speculative-decoding row falls ~6% short of the 1750 threshold: the BF16 draft head's overhead is large relative to the already-cheap FP8 verifier step (TTFT 37 → 65 ms with the draft head), so most of the theoretical `1595 × 1.40 ≈ 2233 tok/s` headroom is absorbed by draft/verify cost. The lever to close it — consistent with the assignment's own guidance and with what moved the BF16 run past its threshold — is a stronger draft head (more training samples → higher acceptance length), ideally trained against the FP8 verifier per the recommendation above.